# 02 — Preprocessing: CRMLS Sold Listings → AVM modeling table

**Input:** raw monthly CSVs in `data/`  
**Output:** `data/processed/sfr_clean.csv` (one row per single-family sale; also saved as `.parquet`) and `docs/cleaning_log.csv`  
**Author:** Ethan Chung — IDX Exchange Data Science, Fall 2026

All logic lives in `src/cleaning.py` so the same rules run from the notebook or the command line
(`python src/cleaning.py`). This notebook runs it, shows what each rule did, and sanity-checks the result.
Thresholds come from the EDA in `01_exploration.ipynb`.

**Scope so far:** cleaning (removing clear errors with fixed thresholds on all data).
Section 7 adds the train / validation / test split and the 0.5th / 99.5th percentile ClosePrice cut (computed on training data only).
Section 8 builds the feature set and the leakage-safe scikit-learn preprocessing pipeline (imputation, missing-value flags, encoding, scaling), fit on training data only.

## 1. Setup

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../src"))
import cleaning
import features

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Rules

| Step | Rule | Why |
|---|---|---|
| Population | `PropertyType == Residential` and `PropertySubType == SingleFamilyResidence`, state CA | AVM scope |
| Duplicates | One row per `ListingKey`, keep latest `ContractStatusChangeDate` | Same sale re-recorded with a corrected CloseDate |
| Timelines | Drop sales with `CloseDate` before `ListingContractDate` or `PurchaseContractDate` | Logically impossible (Best Practices §02); offers dated before the listing are kept, since pre-listing offers happen |
| Target | Drop `ClosePrice` < $10k | Placeholder / error prices |
| Target | Drop close/list ratio outside [0.5, 2] | Catches extra/missing-zero typos (e.g. $970M on a $975k list) |
| Size | Drop `LivingArea` missing or outside 300–20,000 sq ft | Key feature; needed for price/sq ft check |
| Size | Drop price per sq ft outside $50–$5,000 | Remaining size/price mismatches |
| Features | Null impossible values (beds/baths ≤ 0 or > 20, YearBuilt < 1850 or in the future, negative DOM/parking, lots > 1,000 acres, coords outside CA) | Keep the sale, let the model impute the feature |
| Features | Fill `LotSizeSquareFeet` from `LotSizeAcres`, drop acres | One lot-size column |
| Features | Lot "sq ft" under 10 × 43,560 (acres typed in the wrong field); 10–499 sq ft → missing | 337 + 46 lots, mostly San Diego / Riverside |
| Flags | `latfilled` → `coords_imputed`; missing `BasementYN`/`WaterfrontYN` → False | Those two flags only ever record True |
| Columns | Drop PII, agent/office fields, IDs, constant and empty columns | Not property features; buyer-side info is only known after the sale |
| Leakage | Keep `cleaning.LEAKAGE_COLS` in the table but never as features | Required by the IDX Best Practices doc |
| Derived | `close_month`, `log_close_price`, `property_age` | Split key, target, feature |

Thresholds are in `cleaning.RULES`:

In [ ]:
cleaning.RULES

## 3. Run

In [ ]:
raw = cleaning.load_raw("../data")
clean, log = cleaning.clean(raw)
log

In [ ]:
sfr_start = log.loc[log["step"] == "not Residential / SingleFamilyResidence", "rows_after"].item()
print(f"Single-family sales before cleaning: {sfr_start:,}")
print(f"After cleaning:                      {len(clean):,}  ({len(clean) / sfr_start:.1%} kept)")
print(f"Columns: {raw.shape[1]} raw -> {clean.shape[1]} clean")

## 4. Sanity checks on the clean table

In [ ]:
assert clean["ListingKey"].is_unique
assert clean["ClosePrice"].ge(cleaning.RULES["min_close_price"]).all()
assert clean["LivingArea"].between(*cleaning.RULES["living_area_sqft"]).all()
assert not set(cleaning.DROP_COLS) & set(clean.columns)
print("All checks passed")

In [ ]:
clean[["ClosePrice", "LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "YearBuilt",
       "property_age", "LotSizeSquareFeet", "DaysOnMarket", "AssociationFee"]].describe().T

In [ ]:
missing = (clean.isna().mean() * 100).sort_values(ascending=False)
missing[missing > 0].round(1).to_frame("% missing")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(clean["log_close_price"], bins=100)
axes[0].set_title("log(ClosePrice)")
axes[1].hist(clean["ClosePrice"] / clean["LivingArea"], bins=100)
axes[1].set_title("Price per sq ft")
clean.groupby("close_month").size().plot(kind="bar", ax=axes[2], title="Sales per month")
axes[2].tick_params(axis="x", labelsize=6)
plt.tight_layout(); plt.show()

## 5. Save

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
clean.to_csv("../data/processed/sfr_clean.csv", index=False)
clean.to_parquet("../data/processed/sfr_clean.parquet", index=False)
log.to_csv("../docs/cleaning_log.csv", index=False)
print("Saved", clean.shape)

In [ ]:
print("Excluded as features:", cleaning.LEAKAGE_COLS)

## 7. Train / validation / test split and outlier cut

Logic lives in `src/preprocessing.py`, following section 01 of the IDX Best Practices doc.

- **Test** = the latest close month (2026-04), touched only once at the end.
- **Validation** = the month before test (2026-03), used for tuning and for choosing the training window. Never tune on test.
- **Train** = the `train_months` months right before the validation month. The window length is tunable.
- **Outlier cut:** ClosePrice 0.5th / 99.5th percentile, computed on the training set only, then the same frozen cutoffs are applied to validation and test.
  Computing the cutoffs on all data would let held-out prices influence training.

In [ ]:
import preprocessing

windows = preprocessing.compare_windows(clean)
windows

In [ ]:
train, val, test, info = preprocessing.make_split(clean, train_months=preprocessing.DEFAULT_TRAIN_MONTHS)
assert train["close_month"].max() < val["close_month"].min() < test["close_month"].min()
keys = [set(part["ListingKey"]) for part in (train, val, test)]
assert not (keys[0] & keys[1] or keys[0] & keys[2] or keys[1] & keys[2])
info

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
x = windows["train_months"].astype(str)
ax.bar(x, windows["train_rows"])
for j, v in enumerate(windows["train_rows"]):
    ax.text(j, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
ax.set_xlabel("training window (months before the 2026-03 validation month)")
ax.set_ylabel("training rows after cut")
ax.set_title("Training rows by window length")
plt.tight_layout(); plt.show()

**Observations**
- Cutoffs barely move with window length (low ≈ $189k–$200k, high ≈ $8.0M–$8.5M), so the outlier rule is stable.
- About 1% of validation and test sales fall outside the training cutoffs (≈130 each).
- A longer window gives more rows (27k for 3 months vs 264k for 24) but includes older market conditions. The window is chosen by **validation** error once a model exists, never by test error.

## 8. Features and preprocessing pipeline

Logic lives in `src/features.py`, following §05–§07 of the IDX Best Practices doc.

| Group | Columns | Treatment (all fit on training data only) |
|---|---|---|
| Size (skewed) | `LivingArea`, `LotSizeSquareFeet` | median impute + missing flag, log, standardize |
| Numeric | beds, baths, age at sale, garage, parking, stories, main-level beds, HOA per month, lat/lon, `month_sin`/`month_cos` | median impute + missing flag, standardize |
| Yes/No | view, pool, new construction, fireplace, attached garage, basement, waterfront | most-frequent impute + missing flag |
| Low-cardinality | `CountyOrParish`, `Levels` | one-hot; categories with < 50 training rows grouped as "infrequent" |
| High-cardinality | `PostalCode`, `City`, `MLSAreaMajor`, `HighSchoolDistrict` | target encoding, cross-fitted with 5 folds so a row's own price never feeds its own encoding |

**Missingness rule:** columns more than 60% missing in training are dropped (schools, subdivision, builder). Missing-value flags are kept for the rest, because missingness can carry information.

**Assumptions:** an HOA fee with no stated frequency is monthly; for prediction, "month of sale" and "age at sale" use the valuation date.

In [ ]:
audit = features.leakage_audit(clean.columns)
assert (audit["verdict"] != "UNREVIEWED").all()
audit.sort_values(["verdict", "column"])

In [ ]:
X_train, y_train = features.make_xy(train)
X_val, y_val = features.make_xy(val)
X_test, y_test = features.make_xy(test)

pre = features.build_preprocessor()
Xt_train = pre.fit_transform(X_train, y_train)   # the only call that learns from data
Xt_val, Xt_test = pre.transform(X_val), pre.transform(X_test)
print("train", Xt_train.shape, "| val", Xt_val.shape, "| test", Xt_test.shape)
assert not any(np.isnan(a).any() for a in (Xt_train, Xt_val, Xt_test))

In [ ]:
# Leakage checks on the fitted pipeline
names = list(pre.get_feature_names_out())
assert not set(names) & set(cleaning.LEAKAGE_COLS)

# Imputation medians come from training data only
imp = pre.named_transformers_["num"].named_steps["simpleimputer"]
assert np.allclose(imp.statistics_, X_train[features.NUMERIC].median().to_numpy(), equal_nan=True)

# Target encoder learned its categories and global mean from training data only;
# ZIPs never seen in training fall back to that mean
enc = pre.named_transformers_["high_card"].named_steps["targetencoder"]
assert abs(enc.target_mean_ - y_train.mean()) < 1e-9
unseen = ~X_test["PostalCode"].isin(X_train["PostalCode"])
print(f"{len(names)} features; test rows with a ZIP unseen in training: {unseen.sum()}")

## 6. Notes for modeling

- **Target:** `log_close_price`. Report errors back in dollars.
- **Excluded as features** (IDX Best Practices doc): `ListPrice`, `OriginalListPrice`, `DaysOnMarket` and post-listing / post-close dates. The list is `cleaning.LEAKAGE_COLS`.
- **Split:** by `close_month`. 2026-04 = test, 2026-03 = validation; the training window length is tuned on validation.
- **Split + outliers:** done in section 7 (`preprocessing.make_split` returns train, validation and test).
- **Preprocessing:** done in section 8 (`features.build_preprocessor`). Fit it on the training set inside the model pipeline; never on validation or test.
- **Week 6 ideas:** `Flooring` as multi-hot features, bed/bath ratio, school-district spatial join.